In [1]:
import math
from collections import defaultdict

class OptimizedViterbi:
    def __init__(self):
        # States and their indices for faster access
        self.states = ['E', '5', 'I']
        self.state_idx = {s: i for i, s in enumerate(self.states)}
        
        # Transition log probabilities (matrix form)
        self.trans_log = [
            [math.log(0.9), math.log(0.1), float('-inf')],  # E -> E, 5, I
            [float('-inf'), float('-inf'), math.log(1.0)],   # 5 -> E, 5, I
            [float('-inf'), float('-inf'), math.log(0.9)]    # I -> E, 5, I
        ]
        
        # Emission log probabilities (matrix form)
        self.emit_log = [
            [math.log(0.25)] * 4,                          # E emits A,C,G,T
            [math.log(0.05), float('-inf'), math.log(0.95), float('-inf')],  # 5
            [math.log(0.4), math.log(0.1), math.log(0.1), math.log(0.4)]     # I
        ]
        
        # Base to index mapping
        self.base_idx = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
        
        # Initial probabilities
        self.init_log = [0.0, float('-inf'), float('-inf')]  # E, 5, I

    def get_log_prob(self, path, sequence):
        if len(path) != len(sequence):
            return float('-inf')
        
        total = self.init_log[self.state_idx[path[0]]]
        total += self.emit_log[self.state_idx[path[0]]][self.base_idx[sequence[0]]]
        
        for i in range(1, len(path)):
            prev = self.state_idx[path[i-1]]
            curr = self.state_idx[path[i]]
            base = self.base_idx[sequence[i]]
            
            total += self.trans_log[prev][curr]
            total += self.emit_log[curr][base]
        
        # Add termination probability
        total += math.log(0.1)
        
        return total

    def decode(self, sequence):
        """Optimized Viterbi decoding"""
        n = len(sequence)
        m = len(self.states)
        
        # Initialize DP tables
        viterbi = [[float('-inf')] * m for _ in range(n)]
        backptr = [[-1] * m for _ in range(n)]
        
        # First position
        for s in range(m):
            base = self.base_idx[sequence[0]]
            viterbi[0][s] = self.init_log[s] + self.emit_log[s][base]
        
        # Fill tables
        for t in range(1, n):
            base = self.base_idx[sequence[t]]
            for curr in range(m):
                max_prob = float('-inf')
                best_prev = -1
                
                for prev in range(m):
                    prob = viterbi[t-1][prev] + self.trans_log[prev][curr]
                    if prob > max_prob:
                        max_prob = prob
                        best_prev = prev
                
                viterbi[t][curr] = max_prob + self.emit_log[curr][base]
                backptr[t][curr] = best_prev
        
        # Find best path
        best_end = max(range(m), key=lambda s: viterbi[-1][s])
        best_prob = viterbi[-1][best_end]
        
        # Backtrack
        path = [best_end]
        for t in range(n-1, 0, -1):
            path.append(backptr[t][path[-1]])
        path.reverse()
        
        return ''.join(self.states[s] for s in path), best_prob

# Testing
if __name__ == "__main__":
    decoder = OptimizedViterbi()
    
    # Test case from Nature Primer
    path = "EEEEEEEEEEEEEEEEEE5IIIIIII"
    seq = "CTTCATGTGAAAGCAGACGTAAGTCA"
    print(f"Log probability: {decoder.get_log_prob(path, seq):.2f}")  # -41.22
    
    # Viterbi decoding
    best_path, prob = decoder.decode(seq)
    print(f"Most probable path: {best_path}")
    print(f"Log probability: {prob:.2f}")

Log probability: -41.22
Most probable path: EEEEEEEEEEEEEEEEEEEEEEEEEE
Log probability: -38.68
